# Optimización de Hiperparámetros VCP con Optuna

Busca la mejor configuración de ~12 parámetros del detector VCP usando TPE (Tree-structured Parzen Estimator).

**Prerequisitos:**
```bash
pip install optuna plotly mlflow
```

## 1. Setup e imports

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

optuna.logging.set_verbosity(optuna.logging.INFO)
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")

project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from autoresearch import (
    filter_tickers_by_start_date,
    find_common_period,
    get_ticker_info,
    load_universe,
    build_objective_function,
    create_study,
    run_backtest_for_params,
    compute_objective_score,
    study_to_dataframe,
    top_trials_summary,
    param_importance,
    reconstruct_pipeline_params,
    trades_to_dataframe,
    MLflowOptunaLogger,
    DEFAULT_RISK_PARAMS,
)

pd.set_option("display.float_format", "{:.4f}".format)
DATA_DIR = project_root / "data" / "csv"
print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")
print("Imports OK")

## 2. Cargar y filtrar universo de tickers

In [ ]:
ticker_info = get_ticker_info(DATA_DIR)
display(ticker_info)

In [ ]:
opt_tickers, oos_tickers = filter_tickers_by_start_date(
    DATA_DIR, max_start_date="2015-12-31", exclude_tickers=["META"]
)
print(f"Optimization set ({len(opt_tickers)}): {opt_tickers}")
print(f"Out-of-sample set ({len(oos_tickers)}): {oos_tickers}")

In [ ]:
universe = load_universe(opt_tickers, DATA_DIR)
common_start, common_end = find_common_period(universe)
print(f"Universe: {len(universe)} tickers")
print(f"Common period: {common_start.date()} to {common_end.date()}")
print(f"Bars per ticker: {[len(df) for df in universe.values()]}")

## 3. Baseline: parámetros del experimento original

Corremos el pipeline con los parámetros conocidos (`volume_ratio_threshold=1.5`) como referencia.

In [ ]:
from models.configs import ATRZigZagConfig

BASELINE_PARAMS = {
    "swing_config": ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False),
    "sequence_params": {
        "method": "tolerance",
        "min_contractions": 2,
        "max_contractions": 6,
        "lookback_bars": 126,
        "tolerance": 0.10,
        "max_depth_pct": 0.35,
        "min_total_reduction": 0.80,
        "max_gap_between_contractions_days": None,
    },
    "compression_params": {
        "method": "ratio",
        "atr_period": 14,
        "ratio_threshold": 0.85,
    },
    "volume_contraction_params": {
        "method": "ratio",
        "volume_column": "volume",
        "ratio_threshold": 0.85,
    },
    "breakout_params": {
        "volume_method": "ratio",
        "volume_ratio_threshold": 1.5,
        "volume_lookback_days": 50,
        "require_volume_confirmation": True,
    },
    "grouping_params": {"max_gap_days": 30},
    "risk_params": dict(DEFAULT_RISK_PARAMS),
}

In [ ]:
baseline_result = run_backtest_for_params(universe, BASELINE_PARAMS)
baseline_metrics = baseline_result["metrics"]
baseline_score = compute_objective_score(baseline_metrics)

print("=== BASELINE ===")
print(f"  n_trades:       {baseline_metrics['n_trades']}")
print(f"  expectancy_r:   {baseline_metrics['expectancy_r']:+.4f}")
print(f"  win_rate:       {baseline_metrics['win_rate']:.1%}")
print(f"  profit_factor:  {baseline_metrics['profit_factor']:.2f}")
print(f"  score (Opt C):  {baseline_score:+.4f}")
print(f"\n  Trades per ticker:")
for ticker, n in sorted(baseline_metrics["trades_per_ticker"].items()):
    print(f"    {ticker}: {n}")

## 4. Configurar y correr optimización (50 trials)

TPESampler con `multivariate=True`, 20 startup trials random, seed=42.
Cada trial corre el pipeline completo sobre los 18 tickers (~5-25s/trial).

In [ ]:
N_TRIALS = 50
STUDY_NAME = "vcp_optuna_phase1"
MLFLOW_EXPERIMENT = "autoresearch_vcp_phase1"

In [ ]:
objective = build_objective_function(universe)
study = create_study(study_name=STUDY_NAME, n_startup_trials=20, seed=42)
logger = MLflowOptunaLogger(experiment_name=MLFLOW_EXPERIMENT)

with logger.parent_run(study_name=STUDY_NAME, tags={"n_tickers": str(len(universe))}):
    study.optimize(objective, n_trials=N_TRIALS, callbacks=[logger.optuna_callback])
    logger.log_study_summary(study, n_tickers=len(universe))

print(f"\nOptimización completada: {len(study.trials)} trials")
print(f"Best score: {study.best_value:+.4f} (trial #{study.best_trial.number})")

## 5. Inspeccionar resultados

### 5a. Best trial

In [ ]:
best = study.best_trial
print(f"Best trial: #{best.number}")
print(f"  score:         {best.value:+.4f}")
print(f"  n_trades:      {best.user_attrs['n_trades']}")
print(f"  expectancy_r:  {best.user_attrs['expectancy_r']:+.4f}")
print(f"  win_rate:      {best.user_attrs['win_rate']:.1%}")
print(f"  profit_factor: {best.user_attrs['profit_factor']:.2f}")
print(f"\nParams:")
for k, v in sorted(best.params.items()):
    print(f"  {k}: {v}")

### 5b. Top 10 trials

In [ ]:
top10 = top_trials_summary(study, n=10)
display(top10)

### 5c. Importancia de parámetros (fANOVA)

In [ ]:
imp = param_importance(study)
if imp is not None:
    display(imp)
else:
    print("param_importance no pudo calcularse (pocos trials o error de fANOVA)")

### 5d. Visualizaciones de Optuna

In [ ]:
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_parallel_coordinate,
        plot_param_importances,
        plot_slice,
    )
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly no instalado. Correr: pip install plotly")

In [ ]:
if HAS_PLOTLY:
    fig = plot_optimization_history(study)
    fig.update_layout(title="Optimization History", height=400)
    fig.show()

In [ ]:
if HAS_PLOTLY:
    fig = plot_parallel_coordinate(
        study,
        params=[
            "atr_mult", "tolerance", "max_depth_pct",
            "compression_threshold", "vol_contraction_threshold",
            "volume_ratio_threshold",
        ],
    )
    fig.update_layout(title="Parallel Coordinate (top params)", height=500)
    fig.show()

In [ ]:
if HAS_PLOTLY:
    try:
        fig = plot_param_importances(study)
        fig.update_layout(title="Parameter Importances (fANOVA)", height=400)
        fig.show()
    except Exception as e:
        print(f"plot_param_importances falló: {e}")

In [ ]:
if HAS_PLOTLY:
    fig = plot_slice(
        study,
        params=["atr_mult", "tolerance", "compression_threshold", "volume_ratio_threshold"],
    )
    fig.update_layout(title="Slice Plot (key params vs score)", height=400)
    fig.show()

## 6. Re-ejecutar best trial para validar reproducibilidad

In [ ]:
best_params = reconstruct_pipeline_params(study.best_trial.params)
rerun_result = run_backtest_for_params(universe, best_params)
rerun_metrics = rerun_result["metrics"]
rerun_score = compute_objective_score(rerun_metrics)

original_score = study.best_value
score_diff = abs(rerun_score - original_score)

print(f"Score original:    {original_score:+.6f}")
print(f"Score re-run:      {rerun_score:+.6f}")
print(f"Diferencia:        {score_diff:.2e}")

if score_diff > 1e-6:
    print("\n⚠ WARNING: scores difieren. Posible floating point o non-determinism.")
    print(f"  Original n_trades: {study.best_trial.user_attrs['n_trades']}")
    print(f"  Re-run n_trades:   {rerun_metrics['n_trades']}")
else:
    print("\nReproducibilidad OK: scores idénticos.")

In [ ]:
best_trades_df = trades_to_dataframe(rerun_result["all_trades"])
print(f"Trades del best trial: {len(best_trades_df)}")
display(best_trades_df)

## 7. Tabla comparativa: baseline vs best Optuna

In [ ]:
comparison = pd.DataFrame({
    "Baseline (original)": {
        "score": baseline_score,
        "n_trades": baseline_metrics["n_trades"],
        "expectancy_r": baseline_metrics["expectancy_r"],
        "win_rate": baseline_metrics["win_rate"],
        "profit_factor": baseline_metrics["profit_factor"],
        "avg_winner_r": baseline_metrics["avg_winner_r"],
        "avg_loser_r": baseline_metrics["avg_loser_r"],
    },
    "Best Optuna": {
        "score": rerun_score,
        "n_trades": rerun_metrics["n_trades"],
        "expectancy_r": rerun_metrics["expectancy_r"],
        "win_rate": rerun_metrics["win_rate"],
        "profit_factor": rerun_metrics["profit_factor"],
        "avg_winner_r": rerun_metrics["avg_winner_r"],
        "avg_loser_r": rerun_metrics["avg_loser_r"],
    },
}).T

display(comparison)

In [ ]:
print("=== Parámetros del baseline vs best Optuna ===")
print(f"{'Parámetro':<30} {'Baseline':>12} {'Optuna':>12}")
print("-" * 56)

baseline_flat = {
    "atr_length": 14, "atr_mult": 2.0,
    "min_contractions": 2, "max_contractions": 6,
    "lookback_bars": 126, "tolerance": 0.10,
    "max_depth_pct": 0.35, "min_total_reduction": 0.80,
    "compression_threshold": 0.85, "vol_contraction_threshold": 0.85,
    "volume_ratio_threshold": 1.5, "max_gap_days": 30,
}
optuna_flat = study.best_trial.params

for k in baseline_flat:
    bval = baseline_flat[k]
    oval = optuna_flat.get(k, "N/A")
    if isinstance(bval, float):
        print(f"{k:<30} {bval:>12.4f} {oval:>12.4f}")
    else:
        print(f"{k:<30} {bval:>12} {oval:>12}")

## 8. Distribución de trades por ticker (best trial)

In [ ]:
ticker_comparison = []
for ticker in sorted(universe.keys()):
    bl_n = baseline_result["per_ticker"].get(ticker, {}).get("n_trades", 0)
    opt_n = rerun_result["per_ticker"].get(ticker, {}).get("n_trades", 0)
    ticker_comparison.append({"ticker": ticker, "baseline_trades": bl_n, "optuna_trades": opt_n})

tc_df = pd.DataFrame(ticker_comparison).set_index("ticker")
tc_df["diff"] = tc_df["optuna_trades"] - tc_df["baseline_trades"]
display(tc_df)
print(f"\nBaseline total: {tc_df['baseline_trades'].sum()}")
print(f"Optuna total:   {tc_df['optuna_trades'].sum()}")

## 9. Todos los trials (DataFrame completo)

In [ ]:
all_trials_df = study_to_dataframe(study)
print(f"Total trials: {len(all_trials_df)}")
print(f"Trials con score > 0: {(all_trials_df['score'] > 0).sum()}")
print(f"Trials con n_trades == 0: {(all_trials_df['n_trades'] == 0).sum()}")
print(f"Trials con n_trades >= 10: {(all_trials_df['n_trades'] >= 10).sum()}")
display(all_trials_df.head(20))

## 10. Conclusiones y próximos pasos

### Resultados
- **Baseline score** vs **Best Optuna score**: ver tabla comparativa arriba.
- Los parámetros que más impactan el score se ven en la tabla de fANOVA importance.
- La distribución de trades por ticker muestra si Optuna concentra señales en pocos tickers o diversifica.

### Limitaciones de esta Fase 1
- **In-sample optimization**: estamos optimizando y evaluando sobre el mismo periodo. No hay validación out-of-sample.
- **Sin walk-forward**: no probamos si los parámetros generalizan a periodos futuros.
- **Solo `method=tolerance`**: no exploramos `robust_trend` ni otros métodos de compresión.
- **50 trials**: con 12 parámetros, TPE necesita más trials para convergir bien (~200-500).

### Próximos pasos (Fase 2)
1. **Walk-forward validation**: dividir el periodo en ventanas train/test y optimizar rolling.
2. **Out-of-sample test**: correr best params sobre los tickers excluidos (COIN, HOOD, PLTR, SOFI).
3. **Más trials**: escalar a 200+ trials para explorar mejor el espacio.
4. **Explorar métodos**: agregar `robust_trend` y `ratio_normalized` al search space.
5. **Multi-objective**: optimizar expectancy y n_trades como objetivos separados (Pareto front).

### MLflow
Para explorar los resultados en detalle:
```bash
cd /home/gdelarosa/proyectos/deteccion-vcp && mlflow ui --backend-store-uri mlruns/
```